# Memory (대화 내용 기억)
`04_memory.ipynb`
- 대화 내용 기억
- LLM은 기본적으로 대화 내용을 기억하지 않는다(stateless)
- 이전 대화내용을 계속 프롬프트에 주입해야 함

1. Short-Term Memory
    - 단기기억: 한 대화 세션에 대한 기억
2. Long-Term Memory
    - 장기기억: 전체 세션에서 추출한 중요한 정보

## Memory 구동 방식
1. 메모리는 기본적으로 모든 대화 내역을 LLM Input에 밀어 넣는것.
2. 이때 대화가 길어지면 토큰수 증가 및 성능 하락이 일어남
3. 개선 방식 컨셉
    1. 요약
    2. 적정 길이에서 앞부분 자르기
    3. 정리(특정 명사들로 정리, Node-Edge 그래프 방식 등)

## `ConversationBufferMemory`
- 메시지 저장 -> 변수에서 추출 가능

In [2]:
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)

prompt = ChatPromptTemplate(
    [
        ('system', '넌 유용한 챗봇이야'),
        MessagesPlaceholder(variable_name='chat_history'),  # 기존 채팅 내역을 다 주입
        ('human', '{input}'),
    ]
)

memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

# 메모리를 저장할 변수는 {}다. 기존에 대화내용이 있다면 불러와라
memory.load_memory_variables({})

C:\Users\samsung\AppData\Local\Temp\ipykernel_23468\1352095571.py:19: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')


{'chat_history': []}

In [3]:
from operator import itemgetter
# `chat_history`변수에, load_memory_var 결과를 저장하고, `chat_history 키를 추출`
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) |
    itemgetter('chat_history')
)

In [4]:
chain = runnable | prompt | llm 

In [5]:
res = chain.invoke({'input': '만나서 반가워'})

In [6]:
memory.save_context(
    {'human': '만나서 반가워'},
    {'ai': res.content}
)

In [7]:
my_message = '내 이름 뭐라고?'
res = chain.invoke({'input': my_message})
memory.save_context(
    {'human': my_message},
    {'ai': res.content}
)

In [8]:
memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='만나서 반가워', additional_kwargs={}, response_metadata={}),
  AIMessage(content='저도 만나서 반가워요! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='내 이름 뭐라고?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='죄송하지만, 제가 당신의 이름을 알지 못해요. 이름을 알려주시면 기억해둘게요!', additional_kwargs={}, response_metadata={})]}

In [2]:
import time
from operator import itemgetter
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

# LLM
llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)


# Prompt
prompt = ChatPromptTemplate(
    [
        ('system', '넌 좀 틱틱대는 챗봇이야. '),
        MessagesPlaceholder(variable_name='chat_history'),  # 기존 채팅 내역을 다 주입
        ('human', '{input}'),
    ]
)

# Memory
memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) |
    itemgetter('chat_history')
)

chain = runnable | prompt | llm | StrOutputParser()

# 메세지 스트리밍 하는 함수
def stream_msg(chain, msg):
    full_msg = ''
    print('AI: ', end='')
    for token in chain.stream({'input': msg}):
        full_msg += token
        print(token, end='', flush=True)
    print('\n---')
    return full_msg

input_msg = ''

# 사용자가 ('quit', '정지', '그만') 중에 하나를 입력하면 대화 종료
while input_msg not in ('quit', '정지', '그만'):
    input_msg = input()
    print('인간: ', input_msg)
    output_msg = stream_msg(chain, input_msg)
    memory.save_context(
        {'human': input_msg},
        {'ai': output_msg}
    )
    time.sleep(0.5)

C:\Users\samsung\AppData\Local\Temp\ipykernel_27348\3026171462.py:27: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')


인간:  안녕
AI: 안녕? 뭐 궁금한 거 있어?
---
인간:  난 프로그래머야
AI: 오, 프로그래머라니 멋지네! 어떤 언어나 분야에 주로 일해?
---
인간:  알아서 뭐하게?
AI: 그렇구나, 그냥 궁금해서 물어본 거야. 혹시 도움이 필요하거나 이야기하고 싶은 거 있으면 말해줘.
---
인간:  미안해
AI: 괜찮아, 뭐 그럴 수도 있지. 부담 갖지 말고 언제든 이야기하고 싶을 때 말해줘.
---
인간:  내 직업이 뭐야
AI: 네가 프로그래머라고 했으니까, 아마 프로그래머일 거야. 맞아?
---
인간:  그만할까?
AI: 그만두고 싶을 때도 있지. 하지만 네가 원한다면 언제든지 다시 시작할 수 있어. 고민이 있거나 힘든 점 있으면 말해봐. 도움이 될 수 있게 도와줄게.
---
인간:  정지
AI: 알겠어. 필요하면 언제든 돌아와. 잘 지내!
---


In [3]:
# 레포트 작성 챗봇
from operator import itemgetter
from pprint import pprint
from pydantic import BaseModel, Field

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

# LLM
llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)

# Output parser 정의
class Report(BaseModel):
    title: str = Field(..., description='보고서의 제목')
    summary: str = Field(..., description='보고서 요약본')
    content_kr: str = Field(..., description='한국어로 작성된 보고서의 내용(1000자 이내)')
    content_jp: str = Field(..., description='일본어로 작성된 보고서의 내용(1000자 이내)')

parser = PydanticOutputParser(pydantic_object=Report)

# Prompt에 format instructions 추가
prompt = ChatPromptTemplate(
    [
        ('system', '넌 보고서 작성에 특화된 챗봇이야. '
                   '반드시 지정된 형식에 맞춰 JSON으로 답변해.\n\nFORMAT INSTRUCTION: {format_instructions}'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{input}'),
    ]
).partial(format_instructions=parser.get_format_instructions())

# Memory
memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) |
    itemgetter('chat_history')
)

# 체인 구성
chain = runnable | prompt | llm | parser

input_msg = ''

# 사용자가 ('quit', '정지', '그만', '') 중에 하나를 입력하면 대화 종료
while 1:
    input_msg = input()
    if input_msg  in ('quit', '정지', '그만', ''):
        break

    print('인간: ', input_msg)
    output_msg = chain.invoke({'input': input_msg})
    pprint(output_msg.model_dump())
    memory.save_context(
        {'human': input_msg},
        {'ai': output_msg.model_dump_json()}
    )

인간:  신사업 구상
{'content_jp': '本報告書は、新規事業のアイデアと戦略を提案し、企業の成長と競争力の強化を目的としています。市場分析を通じて成長可能性の高い分野を選定し、差別化された価値提案により競争優位を確保する方法を模索します。また、リスク管理と資源配分計画を含め、成功に向けた具体的なロードマップを提示します。これにより、企業は持続可能な成長を図り、市場での地位を強化できます。',
 'content_kr': '본 보고서는 신사업 구상에 대한 구체적인 아이디어와 실행 전략을 제시한다. 시장 분석을 통해 성장 가능성이 높은 '
               '분야를 선정하고, 차별화된 가치 제안을 통해 경쟁 우위를 확보하는 방안을 모색한다. 또한, 리스크 관리와 자원 '
               '배분 계획을 포함하여 성공적인 사업 추진을 위한 구체적인 로드맵을 제시한다. 이를 통해 기업은 지속 가능한 '
               '성장을 도모하고, 시장 내 입지를 강화할 수 있다.',
 'summary': '본 보고서는 새로운 사업 아이디어와 전략을 제시하여 기업의 성장과 경쟁력 강화를 목표로 한다.',
 'title': '신사업 구상'}
